In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

In [0]:
dbutils.widgets.text("p_file_date","2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
#movies_df = spark.read.parquet(f"{silver_folder_path}/movies")
movies_df = spark.read.table("movie_silver.movies")\
                        .filter(f"file_date = '{v_file_date}'")

#genre_df = spark.read.parquet(f"{silver_folder_path}/genre")
genre_df = spark.read.table("movie_silver.genres")

#movie_genres_df = spark.read.parquet(f"{silver_folder_path}/movie_genres")
movie_genres_df = spark.read.table("movie_silver.movie_genres")\
                        .filter(f"file_date = '{v_file_date}'")

In [0]:
movies_join_df = movies_df.join(movie_genres_df,
                                movies_df.movie_id == movie_genres_df.movie_id)\
                            .join(genre_df,
                                  movie_genres_df.genre_id == genre_df.genre_id)\
                            .select("year_release_date",
                                    "budget",
                                    "revenue",
                                    "genre_name")

In [0]:
from pyspark.sql.functions import sum, dense_rank, desc, lit
from pyspark.sql.window import Window

In [0]:
window = Window.partitionBy("year_release_date").orderBy(
                                                        desc("total_budget"), 
                                                        desc("total_revenue")
                                                         )

movies_aggre_df = movies_join_df\
            .filter("year_release_date >= 2015")\
                .groupBy("year_release_date", "genre_name")\
                .agg(
                    sum("budget").alias("total_budget"),
                    sum("revenue").alias("total_revenue")
                         )\
                .withColumn("rank", dense_rank().over(window))\
                .withColumn("created_date", lit(v_file_date))

In [0]:
#overwrite_partition("movie_gold", "results_group_movie_genre", "created_date", v_file_date)

In [0]:
#movies_aggre_df.write.mode("overwrite").parquet(f"{gold_folder_path}/results_group_movie_genre")

#movies_aggre_df.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("movie_gold.results_group_movie_genre")

condition_merge = 'tgt.year_release_date = src.year_release_date AND tgt.genre_name = src.genre_name AND tgt.created_date = src.created_date'

incremental_merge("movie_gold", "results_group_movie_genre", movies_aggre_df, condition_merge, "created_date")

In [0]:
%sql
SELECT * FROM movie_gold.results_group_movie_genre

year_release_date,genre_name,total_budget,total_revenue,rank,created_date
2015,Adventure,2.219E9,8.016412217E9,1,2024-12-30
2015,Action,2.0054E9,7.627777397E9,2,2024-12-30
2015,Drama,1.9518E9,6.473933286E9,3,2024-12-30
2015,Comedy,1.6232E9,6.131404237E9,4,2024-12-30
2015,Thriller,1.42115E9,5.617490431E9,5,2024-12-30
2015,Science Fiction,1.213425E9,4.333796155E9,6,2024-12-30
2015,Family,1.05635E9,4.235493125E9,7,2024-12-30
2015,Crime,9.268E8,1.961292814E9,8,2024-12-30
2015,Animation,7.21E8,3.532697782E9,9,2024-12-30
2015,Fantasy,5.76E8,1.415093823E9,10,2024-12-30


In [0]:
%sql
SELECT created_date, count(1)
FROM movie_gold.results_group_movie_genre
GROUP By created_date

created_date,count(1)
2024-12-23,16
2024-12-30,38
